In [ ]:
from pathlib import Path
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt

DATA_FILE = Path("../data/processed/yellow_tripdata_2025-01_clean.parquet")
FIGURE_DIR = Path("../figures")

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

df = pl.read_parquet(DATA_FILE)

print(df.shape)
df.head()

In [ ]:
summary = df.select([
    pl.len().alias("rows"),
    pl.col("trip_distance").median().alias("median_distance"),
    pl.col("trip_duration_minutes").median().alias("median_duration"),
    pl.col("fare_amount").median().alias("median_fare"),
    pl.col("tip_amount").median().alias("median_tip"),
    pl.col("total_amount").median().alias("median_total"),
])

summary

In [ ]:
hourly = (
    df.group_by("pickup_hour")
    .agg(pl.len().alias("trip_count"))
    .sort("pickup_hour")
    .to_pandas()
)

plt.figure(figsize=(10, 5))
plt.plot(hourly["pickup_hour"], hourly["trip_count"], marker="o")
plt.title("Taxi Trips by Pickup Hour")
plt.xlabel("Pickup Hour")
plt.ylabel("Number of Trips")
plt.xticks(range(24))
plt.tight_layout()

plt.savefig(FIGURE_DIR / "trips_by_hour.png", dpi=150)
plt.show()

In [ ]:
weekday = (
    df.group_by("pickup_weekday")
    .agg(pl.len().alias("trip_count"))
    .sort("pickup_weekday")
    .to_pandas()
)

weekday_names = {
    1: "Mon",
    2: "Tue",
    3: "Wed",
    4: "Thu",
    5: "Fri",
    6: "Sat",
    7: "Sun"
}

weekday["weekday_name"] = weekday["pickup_weekday"].map(weekday_names)

plt.figure(figsize=(8, 5))
plt.bar(weekday["weekday_name"], weekday["trip_count"])
plt.title("Taxi Trips by Weekday")
plt.xlabel("Weekday")
plt.ylabel("Number of Trips")
plt.tight_layout()

plt.savefig(FIGURE_DIR / "trips_by_weekday.png", dpi=150)
plt.show()

In [ ]:
daily = (
    df.group_by("pickup_date")
    .agg(pl.len().alias("trip_count"))
    .sort("pickup_date")
    .to_pandas()
)

daily["pickup_date"] = pd.to_datetime(daily["pickup_date"])

plt.figure(figsize=(12, 5))
plt.plot(daily["pickup_date"], daily["trip_count"], marker="o")
plt.title("Daily Yellow Taxi Trips - January 2025")
plt.xlabel("Date")
plt.ylabel("Number of Trips")
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig(FIGURE_DIR / "daily_trip_counts.png", dpi=150)
plt.show()

In [ ]:
distance_sample = (
    df.filter(pl.col("trip_distance") <= 20)
    .select("trip_distance")
    .sample(n=200_000, seed=42)
    .to_pandas()
)

plt.figure(figsize=(9, 5))
plt.hist(distance_sample["trip_distance"], bins=60)
plt.title("Distribution of Trip Distance")
plt.xlabel("Trip Distance (miles)")
plt.ylabel("Frequency")
plt.tight_layout()

plt.savefig(FIGURE_DIR / "trip_distance_distribution.png", dpi=150)
plt.show()

In [ ]:
duration_sample = (
    df.select("trip_duration_minutes")
    .sample(n=200_000, seed=42)
    .to_pandas()
)

plt.figure(figsize=(9, 5))
plt.hist(duration_sample["trip_duration_minutes"], bins=60)
plt.title("Distribution of Trip Duration")
plt.xlabel("Trip Duration (minutes)")
plt.ylabel("Frequency")
plt.tight_layout()

plt.savefig(FIGURE_DIR / "trip_duration_distribution.png", dpi=150)
plt.show()

In [ ]:
scatter_sample = (
    df.select([
        "trip_distance",
        "fare_amount"
    ])
    .sample(n=200_000, seed=42)
    .to_pandas()
)

plt.figure(figsize=(9, 6))

plt.hexbin(
    scatter_sample["trip_distance"],
    scatter_sample["fare_amount"],
    gridsize=60,
    mincnt=1
)

plt.title("Trip Distance vs Fare Amount")
plt.xlabel("Trip Distance (miles)")
plt.ylabel("Fare Amount ($)")
plt.xlim(0, 30)
plt.ylim(0, 150)

plt.tight_layout()

plt.savefig(FIGURE_DIR / "distance_vs_fare_hexbin.png", dpi=150)
plt.show()

In [ ]:
distance_bins = (
    df.with_columns(
        pl.when(pl.col("trip_distance") <= 1)
        .then(pl.lit("0-1"))
        .when(pl.col("trip_distance") <= 3)
        .then(pl.lit("1-3"))
        .when(pl.col("trip_distance") <= 5)
        .then(pl.lit("3-5"))
        .when(pl.col("trip_distance") <= 10)
        .then(pl.lit("5-10"))
        .otherwise(pl.lit("10+"))
        .alias("distance_group")
    )
    .group_by("distance_group")
    .agg(
        pl.col("fare_amount").mean().alias("avg_fare")
    )
    .to_pandas()
)

order = ["0-1", "1-3", "3-5", "5-10", "10+"]
distance_bins["distance_group"] = pd.Categorical(
    distance_bins["distance_group"],
    categories=order,
    ordered=True
)

distance_bins = distance_bins.sort_values("distance_group")

plt.figure(figsize=(8, 5))
plt.bar(
    distance_bins["distance_group"],
    distance_bins["avg_fare"]
)

plt.title("Average Fare by Trip Distance")
plt.xlabel("Trip Distance Group (miles)")
plt.ylabel("Average Fare ($)")
plt.tight_layout()

plt.savefig(FIGURE_DIR / "avg_fare_by_distance.png", dpi=150)
plt.show()

In [ ]:
top_pickups = (
    df.group_by("PULocationID")
    .agg(pl.len().alias("trip_count"))
    .sort("trip_count", descending=True)
    .head(10)
    .to_pandas()
)

top_pickups["PULocationID"] = top_pickups["PULocationID"].astype(str)

plt.figure(figsize=(9, 5))
plt.bar(top_pickups["PULocationID"], top_pickups["trip_count"])
plt.title("Top 10 Pickup Location IDs")
plt.xlabel("Pickup Location ID")
plt.ylabel("Number of Trips")
plt.tight_layout()

plt.savefig(FIGURE_DIR / "top_pickup_locations.png", dpi=150)
plt.show()

In [ ]:
for file in sorted(FIGURE_DIR.glob("*.png")):
    print(file.name)